# ACIT4620 — E2a-Sensitivity: Longer Fine-Tuning of Class-Weighted EfficientNet-B0

**Group 1 | Follow-up experiment to the completed E2a single-seed run**

This is a **training-schedule sensitivity experiment**, not a replacement of E2a and not a test-set evaluation. Compared with the original E2a: head epochs remain 3, fine-tuning maximum increases from 8 to 20, patience increases from 4 to 5, and patience is tracked **within each stage**. All other primary training settings, seed 42, original HAM10000 audit and lesion-disjoint split, and class-weighted loss are retained.

**Important:** This script independently initializes EfficientNet-B0 from ImageNet weights; do not load E2a's previous checkpoint for training. The same split and validation-only model selection apply. Keep the original E2a saved Kaggle version unchanged.


In [ ]:

from pathlib import Path
import json

files = list(
    Path('/kaggle/input').rglob('audit_summary.json')
)

assert len(files) == 1, f"Found: {files}"

with open(files[0]) as f:
    audit = json.load(f)

for key in [
    'image_count_in_metadata',
    'full_image_audit',
    'images_quality_audited',
    'exact_hash_audit_completed',
    'quality_audit_failures',
    'split_lesion_overlap'
]:
    print(f"{key}: {audit.get(key)}")


## 1. Imports and controls for the E2a training-schedule sensitivity experiment
The sole changes from the first E2a run are longer maximum fine-tuning, increased patience, and stage-specific early-stopping patience. Outputs are saved in a new folder.

In [ ]:
from pathlib import Path
import copy, hashlib, json, os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, accuracy_score, classification_report,
    confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score
)
from scipy.optimize import minimize_scalar
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CLASS_CODES = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
CLASS_TO_IDX = {k: i for i, k in enumerate(CLASS_CODES)}
CLASS_NAMES = {
    'nv':'Melanocytic nevi', 'mel':'Melanoma',
    'bkl':'Benign keratosis-like lesions', 'bcc':'Basal cell carcinoma',
    'akiec':'Actinic keratoses / intraepithelial carcinoma',
    'vasc':'Vascular lesions', 'df':'Dermatofibroma'
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Keep a named folder for each controlled experiment.
EXPERIMENT_NAME = 'efficientnet_b0_classweighted_e2a_long'
USE_CLASS_WEIGHTS = True    # E2a: ONLY controlled intervention relative to E1.
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 32          # Lower to 16 if GPU runs out of memory.
NUM_WORKERS = 2
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
FINETUNE_HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4

REQUIRE_COMPLETE_AUDIT = True   # Blocks training if Notebook 01 audit is missing/incomplete.
RUN_FINAL_TEST = False         # STRICT: do not use test data during E2a development.
USE_TEMPERATURE_SCALING = True # Fit calibration only on validation labels.

OUT = Path('/kaggle/working/efficientnet_runs') / EXPERIMENT_NAME
OUT.mkdir(parents=True, exist_ok=True)
print('Torch:', torch.__version__, '| Device:', DEVICE)
print('Experiment:', EXPERIMENT_NAME)
print('Results will be saved to:', OUT)
if DEVICE.type != 'cuda':
    warnings.warn('No GPU detected. Enable a Kaggle GPU before full training.')

## 2. Locate Notebook 01 outputs — do not create new random splits

**Critical data-centric safeguard:** This notebook reads the *existing* lesion-disjoint manifests. If they are unavailable, it stops and tells you to attach the successful saved Kaggle output. The 10,015 images are the **ISIC 2018 Task 3 training set**, and the held-out test split here is *our own*, not the official challenge test set.

In [ ]:
INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
metadata_files = sorted(INPUT.rglob('HAM10000_metadata.csv'))
if len(metadata_files) != 1:
    raise FileNotFoundError(
        f'Expected ONE raw HAM10000 dataset under Kaggle Input. Found: {metadata_files}. '
        'Attach kmader/skin-cancer-mnist-ham10000 as an input.'
    )
DATA_DIR = metadata_files[0].parent

# Prefer the explicitly labeled seven-class manifest from Notebook 01.
def pick_one(pattern):
    found = sorted(INPUT.rglob(pattern)) + sorted(WORKING.rglob(pattern))
    found = list(dict.fromkeys(str(p.resolve()) for p in found))
    return [Path(p) for p in found]

candidates = pick_one('seven_class_master_manifest.csv')
if not candidates:
    candidates = pick_one('ham10000_master_manifest.csv')
if len(candidates) != 1:
    raise FileNotFoundError(
        'Please attach ONE successfully saved Notebook 01 OUTPUT through Kaggle Add Input. '
        f'Expected a seven-class or master manifest; found: {candidates}. '
        'Do not regenerate random train/test splits.'
    )
MANIFEST_PATH = candidates[0]
manifest = pd.read_csv(MANIFEST_PATH)
print('Raw images at:', DATA_DIR)
print('Verified saved manifest:', MANIFEST_PATH)
print('Available columns:', manifest.columns.tolist())

## 3. Audit quality gates and exact split verification

Full image quality and cross-image hash results should be available from Notebook 01. If not complete, return to Notebook 01, set both audit switches to `True`, rerun, **Save Version**, and attach the new saved output.

In [ ]:
audit_candidates = pick_one('audit_summary.json')
if len(audit_candidates) > 1:
    raise RuntimeError(f'Multiple audit summaries attached; use one saved EDA version: {audit_candidates}')
audit = json.loads(audit_candidates[0].read_text()) if audit_candidates else None
print('Audit summary:', audit_candidates[0] if audit_candidates else 'NOT FOUND')

if audit is not None:
    fields = {k: audit.get(k) for k in [
        'image_count_in_metadata', 'images_quality_audited', 'full_image_audit',
        'exact_hash_audit_completed', 'quality_audit_failures', 'split_lesion_overlap'
    ]}
    print(json.dumps(fields, indent=2))

audit_ok = bool(audit and
                audit.get('full_image_audit') is True and
                audit.get('exact_hash_audit_completed') is True and
                audit.get('images_quality_audited') == 10015 and
                audit.get('quality_audit_failures') == 0 and
                audit.get('split_lesion_overlap') == 0)
if not audit_ok:
    msg = ('Notebook 01 full image/hash audit not confirmed complete. '
           'Check audit_summary.json and investigate any collisions; attach the completed output.')
    if REQUIRE_COMPLETE_AUDIT:
        raise RuntimeError(msg + ' Only for provisional exploration, set REQUIRE_COMPLETE_AUDIT=False.')
    warnings.warn('PROVISIONAL ONLY: ' + msg)

required_cols = {'image_id','lesion_id','dx','split'}
assert required_cols.issubset(manifest.columns), f'Missing: {required_cols - set(manifest.columns)}'
assert len(manifest) == 10015, 'Expected all 10,015 original images.'
assert manifest['image_id'].is_unique, 'Duplicate image ID in manifest.'
assert manifest['lesion_id'].notna().all(), 'Missing lesion ID.'
assert set(manifest['dx']) == set(CLASS_CODES), 'Diagnosis label mismatch.'
assert set(manifest['split']) == {'train','validation','test'}, 'Missing split.'
assert manifest.groupby('lesion_id')['split'].nunique().max() == 1, 'LEAKAGE: repeated lesion crosses splits!'
manifest['label'] = manifest['dx'].map(CLASS_TO_IDX).astype(int)
if 'label' in manifest.columns:
    assert manifest['label'].between(0,6).all()

observed = pd.crosstab(manifest['dx'], manifest['split']).reindex(
    index=CLASS_CODES, columns=['train','validation','test'], fill_value=0)
expected = pd.DataFrame({
    'train':[4796,782,772,358,230,89,89],
    'validation':[974,146,168,89,34,21,9],
    'test':[935,185,159,67,63,32,17]}, index=CLASS_CODES)
display(observed)
assert observed.equals(expected), (
    'Different split assignment from the verified Notebook 01 run. Stop and investigate.')

# Fingerprint the sorted assignments: supports reproducibility across model runs.
assignment_string = '\n'.join(
    manifest[['image_id','lesion_id','dx','split']].astype(str)
        .sort_values('image_id').agg('|'.join, axis=1)
)
SPLIT_SHA256 = hashlib.sha256(assignment_string.encode()).hexdigest()
print('Split assignment SHA-256:', SPLIT_SHA256)
print('PASS: seven labels, correct split counts and zero lesion overlap.')

### Optional E1 split-hash cross-check
If the original E1 `.pth` checkpoint is attached as a Kaggle Input, this cell checks its saved `split_sha256` against E2a. If not attached, the full expected split counts and lesion-ID checks above still protect the split definition. Never use an E1 checkpoint to initialize E2a weights: E2a must use the same ImageNet starting point as E1.

In [ ]:
e1_candidates = sorted(INPUT.rglob('best_efficientnet_b0.pth'))
if len(e1_candidates) == 1:
    baseline_checkpoint = torch.load(e1_candidates[0],map_location='cpu',weights_only=True)
    if 'split_sha256' not in baseline_checkpoint:
        raise RuntimeError('E1 checkpoint lacks split_sha256, cannot verify comparability.')
    assert baseline_checkpoint['split_sha256'] == SPLIT_SHA256, 'E1/E2a split mismatch!'
    print('PASS: E1 checkpoint and E2a manifest use identical lesion-group split.')
elif len(e1_candidates) > 1:
    print('Multiple E1 checkpoint files found; skip optional comparison. Remove duplicates to verify.')
else:
    print('E1 checkpoint not attached; exact split counts + lesion overlap already verified.')

## 4. Reconstruct **actual Kaggle image paths** (never train from metadata labels)

We use `image_relpath` where available because Notebook 01 may have saved absolute paths from an earlier Kaggle session. Image ID and lesion ID are for lookup and leakage prevention only—not CNN inputs.

In [ ]:
# Prefer relpath (saved by Notebook 01); if absent or moved, use image_id from the original folders.
original_image_paths = {}
for folder in (DATA_DIR / 'HAM10000_images_part_1', DATA_DIR / 'HAM10000_images_part_2'):
    if folder.exists():
        for p in folder.glob('*.jpg'):
            if p.stem in original_image_paths:
                raise RuntimeError(f'Duplicate file ID across original image folders: {p.stem}')
            original_image_paths[p.stem] = p
assert len(original_image_paths) == 10015, 'Could not locate all 10,015 original jpg files.'

def resolve_path(row):
    if 'image_relpath' in row.index and pd.notna(row['image_relpath']):
        candidate = DATA_DIR / str(row['image_relpath'])
        if candidate.is_file() and candidate.stem == row['image_id']:
            return str(candidate)
    return str(original_image_paths[row['image_id']])

manifest['resolved_path'] = manifest.apply(resolve_path, axis=1)
assert manifest['resolved_path'].map(lambda p: Path(p).is_file()).all()
assert manifest['resolved_path'].map(lambda p: Path(p).stem).eq(manifest['image_id']).all()
parts = {k: manifest[manifest['split']==k].copy().reset_index(drop=True)
         for k in ['train','validation','test']}
print('Resolved all image files: ', len(manifest))
print('Train / validation / test:', [len(parts[k]) for k in parts])
manifest[['image_id','lesion_id','dx','label','split']].to_csv(OUT / 'split_manifest_summary.csv', index=False)

## 5. Visualize class imbalance and record the majority-class baseline

Avoid judging performance from accuracy alone. Majority-class accuracy on the original seven-class training distribution is ~67%, but balanced accuracy is only 1/7 (14.3%) for an always-`nv` classifier.

In [ ]:
count_train = parts['train']['dx'].value_counts().reindex(CLASS_CODES, fill_value=0)
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(CLASS_CODES, count_train.values)
ax.set(title='HAM10000 training images by class', xlabel='Diagnosis', ylabel='Images')
for i, count in enumerate(count_train.values):
    ax.text(i, count + 35, str(count), ha='center', fontsize=9)
fig.tight_layout(); fig.savefig(OUT / '01_train_imbalance.png', dpi=170); plt.show()
print('Majority-only training accuracy:', round(count_train.max()/len(parts['train']), 4))
print('Majority-only seven-class balanced accuracy:', round(1/7, 4))

## 6. Image preprocessing and train-only augmentation

Resize preserving aspect ratio, then center-crop to 224×224; apply ImageNet normalization. Training-only changes are modest flips, rotations, and color jitter. These augmentations are not automatically clinically valid, so examine a contact sheet and document artifacts before experiments. **No augmentation in validation/test.**

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.06, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

class HAM10000Dataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        item = self.frame.iloc[idx]
        with Image.open(item['resolved_path']) as img:
            img = ImageOps.exif_transpose(img).convert('RGB')
            tensor = self.transform(img)
        return tensor, int(item['label']), item['image_id']

train_ds = HAM10000Dataset(parts['train'], train_transform)
val_ds = HAM10000Dataset(parts['validation'], eval_transform)
test_ds = HAM10000Dataset(parts['test'], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

# Visual sanity check: denormalize a *few* transformed images.
sample_images, sample_labels, _ = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(11,6))
for ax, img, lab in zip(axes.ravel(), sample_images[:8], sample_labels[:8]):
    arr = img.numpy().transpose(1,2,0) * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    ax.imshow(np.clip(arr,0,1)); ax.set_title(CLASS_CODES[int(lab)]); ax.axis('off')
plt.tight_layout(); fig.savefig(OUT / '02_training_transforms_preview.png', dpi=130); plt.show()

## 7. Build **genuinely pretrained** EfficientNet-B0

Use ImageNet-pretrained weights, then replace the classifier with **7 logits**. Stage 1 trains just the classification head; Stage 2 fine-tunes the last two feature blocks and the head. If downloading ImageNet weights fails, enable Kaggle Internet (if available) or add legally obtained pretrained weights to input; **do not silently replace with random initialization**.

In [ ]:
try:
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)
except Exception as exc:
    raise RuntimeError(
        'ImageNet-pretrained EfficientNet-B0 weights could not be loaded. '
        'Check Kaggle Internet permissions or provide cached pretrained weights. '
        'Do not use weights=None as a substitute.'
    ) from exc

n_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(n_features, len(CLASS_CODES))
model = model.to(DEVICE)
for param in model.features.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
model.eval()  # Avoid modifying pretrained BatchNorm running statistics.
with torch.no_grad():
    assert model(torch.zeros(1,3,224,224, device=DEVICE)).shape == (1,7)
print('Output neurons:', len(CLASS_CODES), '| Trainable now:', trainable, '/', total)
print('Pretraining:', weights)

## 8. E2a training-only inverse-frequency weights
Weights use only **training** counts, `N_train / (7*n_class)`. E1 used the same calculation but turned weighting off. Validation class distributions remain unchanged.

In [ ]:
train_counts = np.array([int((parts['train']['dx']==k).sum()) for k in CLASS_CODES])
class_weights = len(parts['train']) / (7.0 * train_counts)
print('Counts:', dict(zip(CLASS_CODES, train_counts.tolist())))
print('Balanced weights:', dict(zip(CLASS_CODES, np.round(class_weights,3))))
weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
assert USE_CLASS_WEIGHTS is True, 'E2a requires weighted cross-entropy.'
assert len(train_counts)==7 and train_counts.sum()==7116
assert np.isfinite(class_weights).all() and (class_weights>0).all()
assert np.isclose(np.dot(train_counts, class_weights)/len(parts['train']),1.0)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
print('E2a criterion: weighted CrossEntropyLoss (only controlled change)')


def compute_metrics(labels, probs):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs)
    pred = probs.argmax(axis=1)
    return {
        'accuracy': float(accuracy_score(labels, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(labels, pred)),
        'macro_f1': float(f1_score(labels, pred, labels=list(range(7)), average='macro', zero_division=0)),
        'melanoma_recall': float(((pred == CLASS_TO_IDX['mel']) & (labels == CLASS_TO_IDX['mel'])).sum() /
                                  max(1, (labels == CLASS_TO_IDX['mel']).sum())),
    }

@torch.no_grad()
def evaluate_loader(loader, include_logits=False):
    model.eval()
    totals, all_logits, all_y, all_ids = 0., [], [], []
    for x,y,img_ids in loader:
        x=x.to(DEVICE, non_blocking=True); y=y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type=='cuda')):
            logits=model(x); loss=criterion(logits,y)
        totals+=float(loss.item())*y.size(0)
        all_logits.append(logits.float().cpu().numpy())
        all_y.append(y.cpu().numpy()); all_ids.extend(img_ids)
    logits=np.concatenate(all_logits)
    y=np.concatenate(all_y)
    probs=torch.softmax(torch.from_numpy(logits),dim=1).numpy()
    result={'loss':totals/len(loader.dataset), **compute_metrics(y,probs)}
    return (result, logits, y, all_ids) if include_logits else result

## 9. Train two stages with **stage-local patience and global best-checkpoint selection**

The original E2a run stopped after four fine-tuning epochs because it compared improvements against the earlier head-stage best. Here the **patience counter only compares within the current stage**. A distinct, global best still determines the single saved checkpoint using validation macro-F1 (balanced accuracy for ties). A fine-tuning epoch that is better than previous fine-tuning epochs but worse than the head-stage best resets stage patience **without overwriting the better global checkpoint**.

The model starts fine-tuning from the head-stage best checkpoint. After fine-tuning, the best checkpoint over both stages is restored. The held-out test remains unused.

In [ ]:
history=[]
BEST_FILE = OUT / 'best_efficientnet_b0.pth'
best_score=(-1.0,-1.0)
AMP = DEVICE.type=='cuda'
scaler = torch.amp.GradScaler('cuda',enabled=AMP)

def run_stage(stage, epochs, optimizer):
    global best_score
    stale=0
    stage_best=(-1.0,-1.0)
    for epoch in range(1, epochs+1):
        model.train()
        # Frozen backbone BatchNorm remains in evaluation mode in the head-only stage.
        if stage=='head':
            model.features.eval()
        else:
            # Frozen early blocks must not update BatchNorm statistics.
            for frozen_block in model.features[:-2]:
                frozen_block.eval()
        total_loss=0.
        for x,y,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                logits=model(x)
                loss=criterion(logits,y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item())*y.size(0)
        train_loss=total_loss/len(train_loader.dataset)
        vm=evaluate_loader(val_loader)
        row={'stage':stage, 'epoch':epoch, 'train_loss':train_loss,
             **{f'val_{k}':v for k,v in vm.items()}}
        history.append(row)
        candidate=(vm['macro_f1'],vm['balanced_accuracy'])
        # Early stopping tracks progress against the best result WITHIN this stage.
        # The global best checkpoint is tracked separately and never overwritten
        # by a worse fine-tuning result.
        stage_improved = candidate[0] > stage_best[0] + 1e-4 or (
            abs(candidate[0]-stage_best[0]) <= 1e-4
            and candidate[1] > stage_best[1] + 1e-4)
        if stage_improved:
            stage_best = candidate
            stale = 0
        else:
            stale += 1

        improved=candidate[0]>best_score[0]+1e-4 or (
            abs(candidate[0]-best_score[0])<=1e-4 and candidate[1]>best_score[1]+1e-4)
        if improved:
            best_score=candidate
            torch.save({'model_state':model.state_dict(), 'class_codes':CLASS_CODES,
                        'val_macro_f1':vm['macro_f1'], 'val_balanced_accuracy':vm['balanced_accuracy'],
                        'epoch':epoch, 'stage':stage, 'split_sha256':SPLIT_SHA256},BEST_FILE)
        print(f'{stage} {epoch:02d}/{epochs} | train_loss={train_loss:.4f} '
              f'val_loss={vm["loss"]:.4f} val_macro_f1={vm["macro_f1"]:.4f} '
              f'val_bal_acc={vm["balanced_accuracy"]:.4f} '
              f'mel_recall={vm["melanoma_recall"]:.4f}'
              + (' *global best*' if improved else (' *stage best*' if stage_improved else '')))
        pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
        if stale>=PATIENCE:
            print('Stage-local early stopping in', stage, 'after', PATIENCE, 'epochs without stage improvement'); break
    # Always continue from the best checkpoint, not the last epoch.
    ckpt=torch.load(BEST_FILE,map_location=DEVICE,weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    print('Restored global best checkpoint after stage:', stage, '| best:', best_score)

head_optimizer=torch.optim.AdamW(model.classifier.parameters(),lr=HEAD_LR,
                                  weight_decay=WEIGHT_DECAY)
run_stage('head',HEAD_EPOCHS,head_optimizer)

# Fine-tune only the final two EfficientNet feature modules.
for p in model.features[-2:].parameters(): p.requires_grad=True
finetune_optimizer=torch.optim.AdamW([
    {'params':model.features[-2:].parameters(), 'lr':BACKBONE_LR},
    {'params':model.classifier.parameters(), 'lr':FINETUNE_HEAD_LR}],
    weight_decay=WEIGHT_DECAY)
run_stage('fine_tune',FINETUNE_EPOCHS,finetune_optimizer)

print('Best validation (macro-F1, balanced acc):',best_score)
print('Checkpoint:',BEST_FILE)

## 10. Visualize the loss and validation metrics

These results are **not invented**: plots are generated only after Kaggle trains the model. Check overfitting (training loss decreasing while validation loss rises), and small-class recall before choosing hyperparameters.

In [ ]:
hist = pd.DataFrame(history)
fig, axs = plt.subplots(1,2,figsize=(12,4))
axs[0].plot(hist.index+1, hist.train_loss, marker='o',label='Train')
axs[0].plot(hist.index+1, hist.val_loss,marker='o',label='Validation')
axs[0].set(title='Loss per completed epoch',xlabel='Epoch',ylabel='Cross-entropy'); axs[0].legend()
axs[1].plot(hist.index+1,hist.val_macro_f1,marker='o',label='Macro-F1')
axs[1].plot(hist.index+1,hist.val_balanced_accuracy,marker='o',label='Balanced accuracy')
axs[1].plot(hist.index+1,hist.val_melanoma_recall,marker='o',label='Melanoma recall')
axs[1].set(title='Validation metrics',xlabel='Epoch',ylabel='Score'); axs[1].legend()
for ax in axs: ax.grid(alpha=0.15)
fig.tight_layout(); fig.savefig(OUT/'03_training_curves.png',dpi=170); plt.show()

## 11. Inspect the **validation** confusion matrix and class-wise errors

Do not touch the test set during model selection. Make data-centric improvements by inspecting **training/validation** failures and image artifacts, then rerun the *same* split. The smallest classes have limited validation examples (e.g. `df`: 9), so point estimates are unstable.

In [ ]:
ckpt=torch.load(BEST_FILE,map_location=DEVICE,weights_only=True)
assert ckpt['class_codes']==CLASS_CODES and ckpt['split_sha256']==SPLIT_SHA256
model.load_state_dict(ckpt['model_state'])
val_metrics,val_logits,val_y,val_ids=evaluate_loader(val_loader,include_logits=True)
val_probs=torch.softmax(torch.from_numpy(val_logits),dim=1).numpy()
val_pred=val_probs.argmax(axis=1)
print('Validation only:',val_metrics)
print(classification_report(val_y,val_pred,labels=range(7),target_names=CLASS_CODES,zero_division=0))
cm=confusion_matrix(val_y,val_pred,labels=range(7))
fig,ax=plt.subplots(figsize=(8,7))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CLASS_CODES,
            yticklabels=CLASS_CODES,ax=ax)
ax.set(xlabel='Predicted diagnosis',ylabel='True diagnosis',title='VALIDATION confusion matrix')
fig.tight_layout(); fig.savefig(OUT/'04_val_confusion_matrix.png',dpi=170); plt.show()

val_frame=parts['validation'].set_index('image_id').loc[val_ids].reset_index()
assert (val_frame['label'].to_numpy()==val_y).all()
val_errors=val_frame[['image_id','lesion_id','dx','label']].copy()
val_errors['predicted_dx']=[CLASS_CODES[i] for i in val_pred]
val_errors['correct']=val_y==val_pred
val_errors['predicted_confidence']=val_probs.max(axis=1)
val_errors['melanoma_probability']=val_probs[:,CLASS_TO_IDX['mel']]
val_errors.to_csv(OUT/'validation_error_analysis.csv',index=False)
print('Validation mistakes:',int((~val_errors.correct).sum()))
display(val_errors.loc[~val_errors.correct].head(20))

# E2a report-ready per-class validation summary (no test data)
pvals, rvals, fvals, supports = precision_recall_fscore_support(
    val_y, val_pred, labels=list(range(7)), zero_division=0)
class_table = pd.DataFrame({
    'class': CLASS_CODES, 'support':supports,
    'precision':pvals, 'recall':rvals, 'f1':fvals})
for c,k in enumerate(CLASS_CODES):
    positive = (val_y==c).astype(int)
    class_table.loc[c,'ovr_roc_auc'] = float(roc_auc_score(positive,val_probs[:,c]))
    class_table.loc[c,'ovr_pr_auc_ap'] = float(average_precision_score(positive,val_probs[:,c]))
class_table.to_csv(OUT/'e2a_validation_per_class.csv',index=False)
E1_PREVIOUS = {'accuracy':0.7509, 'macro_f1':0.4554,
               'balanced_accuracy':0.4436, 'melanoma_recall':29/146}
comparison = pd.DataFrame([
    {'metric':m, 'E1_proposal_reported':v, 'E2a_validation':val_metrics[m],
     'E2a_minus_E1':val_metrics[m]-v}
    for m,v in E1_PREVIOUS.items()
])
comparison.to_csv(OUT/'e1_vs_e2a_validation.csv',index=False)
(OUT/'e2a_validation_metrics.json').write_text(json.dumps({
    'experiment':'E2a', 'split':'validation', 'metrics':val_metrics,
    'e1_reference':'rounded values reported in submitted proposal; E1 macro_f1=0.4554',
    'split_sha256':SPLIT_SHA256},indent=2))
print('E1 vs E2a: validation only; E1 values rounded from the proposal')
display(comparison)
display(class_table)


## 12. Optional probability calibration on validation only

Temperature scaling uses validation logits/labels to fit **one scalar**. This is not an independent calibration dataset: do not report post-calibration validation loss as unbiased generalization evidence. **Never optimize temperature or thresholds on test labels.** Multiclass entropy from the resulting calibrated probabilities will be used later by Mamdani.

In [ ]:
def softmax_np(logits):
    logits=np.asarray(logits,dtype=np.float64)
    z=logits-logits.max(axis=-1,keepdims=True)
    ez=np.exp(z)
    return ez/ez.sum(axis=-1,keepdims=True)

def nll_with_temperature(log_temp, logits, labels):
    T=float(np.exp(log_temp))
    p=softmax_np(logits/T)
    return float(-np.log(np.clip(p[np.arange(len(labels)),labels],1e-12,1)).mean())

if USE_TEMPERATURE_SCALING:
    res=minimize_scalar(nll_with_temperature, bounds=(np.log(0.5),np.log(5.0)),
                        method='bounded',args=(val_logits,val_y))
    TEMPERATURE=float(np.exp(res.x))
else:
    TEMPERATURE=1.0
print('Calibration temperature (from validation only):',round(TEMPERATURE,4))
print('Validation NLL before:',round(nll_with_temperature(0,val_logits,val_y),4))
print('Validation NLL after: ',round(nll_with_temperature(np.log(TEMPERATURE),val_logits,val_y),4))
with (OUT/'temperature_calibration.json').open('w') as f:
    json.dump({'temperature':TEMPERATURE,'fitted_on':'validation',
               'warning':'Not an independent calibration cohort'},f,indent=2)

## 13. Optional original-view + TTA prediction exports (validation only)
Retained from E1 to preserve the recording protocol. These exports are **not a Mamdani inference system** and are not used to select E2a based on the held-out test set. Main performance comparisons above use original-view logits without temperature scaling or TTA.

In [ ]:
TTA_NAMES=['identity','hflip','vflip','rot180']

def view_tensor(x,view):
    if view=='identity': return x
    if view=='hflip': return torch.flip(x,dims=(-1,))
    if view=='vflip': return torch.flip(x,dims=(-2,))
    if view=='rot180': return torch.rot90(x,k=2,dims=(-2,-1))
    raise KeyError(view)

@torch.no_grad()
def predict_for_fuzzy(loader, source_frame, temperature):
    model.eval()
    logits_views=[]
    y_list=[]; image_ids=[]
    for x,y,ids in loader:
        x=x.to(DEVICE,non_blocking=True)
        view_logits=[]
        for v in TTA_NAMES:
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type=='cuda')):
                pred=model(view_tensor(x,v))
            view_logits.append(pred.float().cpu().numpy())
        logits_views.append(np.stack(view_logits,axis=1))
        y_list.append(y.numpy()); image_ids.extend(ids)
    all_logits=np.concatenate(logits_views,axis=0)   # (N,4,7)
    all_y=np.concatenate(y_list)
    assert all_logits.shape==(len(source_frame),4,7)
    ordered=source_frame.set_index('image_id').loc[image_ids]
    assert (ordered['label'].to_numpy()==all_y).all()
    tta_probs=softmax_np(all_logits/temperature).astype(np.float32)
    probs=tta_probs[:,0,:]  # original view for primary CNN prediction
    assert np.allclose(tta_probs.sum(axis=2),1,atol=1e-5)
    return {
        'y_true':all_y.astype(np.int64),
        'probs':probs,
        'tta_probs':tta_probs,
        'lesion_id':ordered['lesion_id'].astype(str).to_numpy(),
        'image_id':np.asarray(image_ids,dtype=str)
    }

def save_predictions(split_name, loader, frame):
    d=predict_for_fuzzy(loader,frame,TEMPERATURE)
    np.savez_compressed(OUT/f'{split_name}_predictions.npz',**d)
    p=d['probs']; true=d['y_true']
    table=pd.DataFrame({'image_id':d['image_id'],'lesion_id':d['lesion_id'],
                        'true_dx':[CLASS_CODES[i] for i in true],
                        'predicted_dx':[CLASS_CODES[i] for i in p.argmax(axis=1)]})
    for k,cls in enumerate(CLASS_CODES): table[f'prob_{cls}']=p[:,k]
    table.to_csv(OUT/f'{split_name}_predictions.csv',index=False)
    return d

val_export=save_predictions('validation',val_loader,parts['validation'])
print('Saved validation predictions with original + 4 TTA views:')
print((OUT/'validation_predictions.npz').name, val_export['tta_probs'].shape)

## 14. Locked test remains untouched
`RUN_FINAL_TEST=False`. **Do not turn it on for E2a model selection.** Final locked testing must occur only after all model selection for E1/E2/E3/E4 is complete.

In [ ]:
assert RUN_FINAL_TEST is False, 'Do not evaluate the final test set in E2a.'
print('PASS: E2a uses only training and validation; held-out test remains locked.')

## 15. Reproducibility manifest and output files
Save this notebook using **Save Version → Save & Run All**, preserving the original E2a Kaggle version. This follow-up is a validation-only *training-schedule sensitivity experiment*. Make a fresh input/output version for each run. No locked test evaluation.

In [ ]:
settings = {
    'experiment':EXPERIMENT_NAME, 'seed':SEED, 'class_codes':CLASS_CODES,
    'split_sha256':SPLIT_SHA256, 'pretraining':'EfficientNet_B0_Weights.DEFAULT (ImageNet)',
    'batch_size':BATCH_SIZE,'head_epochs':HEAD_EPOCHS,'finetune_epochs':FINETUNE_EPOCHS,
    'head_lr':HEAD_LR,'backbone_lr':BACKBONE_LR,'fine_tune_head_lr':FINETUNE_HEAD_LR,
    'use_class_weights':USE_CLASS_WEIGHTS,'class_weights':dict(zip(CLASS_CODES,[float(v) for v in class_weights])),'head_selection':'validation_macro_f1',
    'early_stopping':'stage_local_patience_global_best_checkpoint', 'patience':PATIENCE,
    'experiment_type':'E2a training-schedule sensitivity, single-seed, validation-only',
    'audit_gate_complete':audit_ok,'test_evaluated':RUN_FINAL_TEST,
    'calibration_temperature':TEMPERATURE,'TTA':TTA_NAMES,
    'device':str(DEVICE),'torch':torch.__version__
}
(OUT/'experiment_config.json').write_text(json.dumps(settings,indent=2))
print('Experiment files:')
for p in sorted(OUT.iterdir()):
    print(' -',p.name, f'({p.stat().st_size/1024:.0f} KB)')
print('Remember Kaggle Save Version → Save & Run All.')

## After this sensitivity run

Compare the original E2a and this **E2a-Long** run on the same validation split, documenting both and **not** treating the best of multiple validation trials as unbiased test evidence. Preserve original E1/E2a configurations. For RQ2, select one backbone using validation only and give the linear, MLP and Sugeno ANFIS heads the same frozen 1,280-D EfficientNet-B0 features. Then train E3 and E4 with ANFIS R ∈ {4, 8, 16}; complete repeated-seed, XAI and efficiency work before the single locked test assessment.